#BASIC

1. Use MERGE INTO to upsert a batch of 'changed customer' records into a silver customers table.
2. Grant SELECT on a table to one group and a masked/limited view to another group using Unity
Catalog permissions.
3. Look up the DBU consumption for a compute resource you've been using and explain, in plain terms,
what a DBU is billing for.__

In [0]:
cust =[
(2,"Jane Smith","jane.smith@example.com","Los Angeles","CA",'2022-02-01',9876543210),
(3,"Alice Johnson","alice.johnson@example.com","Chicago","IL",'2022-03-01',5551234567),
(4,"Bob Brown","bob.brown@example.com","San Francisco","CA",'2022-04-01',5559876543),
(5,"Charlie Davis","charlie.davis@example.com","Houston","TX",'2022-05-01',5551112222),
(6,"John Doe","john.doe@example.com","New York","NY",'2022-01-01',1234567890)
]

In [0]:
from pyspark.sql.functions import to_date

schema = ["customer_id","name","email","city","state","signup_date","phone"]
df = spark.createDataFrame(cust, schema)

df = df.withColumn(
    "signup_date",
    to_date("signup_date")
)
df.createOrReplaceTempView("changed_customers")

In [0]:
%sql
MERGE INTO cyntexa_dev.sales.customer_clean AS t
USING changed_customers AS s
ON t.customer_id = s.customer_id

WHEN MATCHED THEN
  UPDATE SET
    t.name = s.name,
    t.email = s.email,
    t.city = s.city,
    t.state = s.state,
    t.signup_date = s.signup_date,
    t.phone = s.phone

WHEN NOT MATCHED THEN
  INSERT (customer_id,name,email,city,state,signup_date,phone)
  VALUES (
    s.customer_id,
    s.name,
    s.email,
    s.city,
    s.state,
    s.signup_date,
    s.phone
  );

In [0]:
%sql
-- 2
GRANT SELECT ON TABLE cyntexa_dev.sales.customer_clean TO `Data_engineer`;

3.

DBU is a unit of compute capacity in compute.
its pricing is based on the :

- workload type - type of compute used and the workloads like jobs,pipeline and more decide the DBUs unit consumption.
- node type - larger GPU consumers more units compare to small nodes.
- pricing tier - the pricing also depends on the tier plan - standard,premium,
- cloud provider - AWS,Azure and GCP has differnt cost plan.

#INTERMEDIATE

In [0]:
%sql
-- 4
CREATE OR REPLACE  TABLE cyntexa_dev.sales.customer_address_history (
    customer_id BIGINT,
    email STRING,
    city STRING,
    state STRING,
    status STRING,
    effective_date DATE,
    end_date DATE,
    is_current BOOLEAN,
    version INT
)
USING DELTA;

In [0]:
from pyspark.sql.functions import to_date

data = [
    (101, "abc@gmail.com","Mumbai", "Maharashtra","Active" ,"2026-03-01"),
    (102, "sdf@gmail.com" ,"Delhi", "Delhi", "Active","2026-03-01")
]

columns = [
    "customer_id",
    "email",
    "city",
    "state",
    "status",
    "effective_date"
]
df = spark.createDataFrame(data, columns)
df = df.withColumn(
    "effective_date",
    to_date("effective_date")
)
df.createOrReplaceTempView("customer_address_source")

In [0]:
%sql
MERGE INTO cyntexa_dev.sales.customer_address_history AS target
USING customer_address_source AS source
ON target.customer_id = source.customer_id
   AND target.is_current = true

WHEN MATCHED
AND (
    target.city != source.city
    OR target.state != source.state
)
THEN UPDATE SET
    target.end_date = source.effective_date,
    target.is_current = false;

In [0]:
%sql
INSERT INTO cyntexa_dev.sales.customer_address_history
(
    customer_id,
    email,
    city,
    state,
    status,
    effective_date,
    end_date,
    is_current,
    version
)
SELECT
    source.customer_id,
    source.email,
    source.city,
    source.state,
    source.status,
    source.effective_date,
    NULL,
    true,
    COALESCE(MAX(target.version), 0) + 1
FROM customer_address_source AS source
LEFT JOIN cyntexa_dev.sales.customer_address_history AS target
ON source.customer_id = target.customer_id

WHERE NOT EXISTS (
    SELECT 1
    FROM cyntexa_dev.sales.customer_address_history AS current_record
    WHERE current_record.customer_id = source.customer_id
      AND current_record.is_current = true
      AND current_record.email = source.email
      AND current_record.city = source.city
      AND current_record.state = source.state
      AND current_record.status = source.status
)
GROUP BY
    source.customer_id,
    source.email,
    source.city,
    source.state,
    source.status,
    source.effective_date;

In [0]:
%sql
-- 5
SELECT customer_id, city, state
FROM cyntexa_dev.sales.customer_address_history
WHERE customer_id = 101
  AND DATE('2026-03-01') >= effective_date
  AND (
        end_date IS NULL
        OR DATE('2026-03-01') < end_date
      );

-- 6
- Cyntexa must use Job Compute clusters for its nightly pipeline.
- because :
- All-Purpose clusters are priced at a much higher DBU rate because they are designed for interactive and analysis. Job Compute clusters run production workloads automatically at roughly half the DBU cost, saving significant cloud spend over time.

#ADVANCED

8. Extend the SCD Type 2 pattern to track changes across 3+ columns simultaneously, and handle the
edge case of a customer record that hasn't changed since the last load (it should not create a false
new version).


## 7- Data Governance Model for Cyntexa

1. Unity Catalog Namespace

Cyntexa follows the three-level Unity Catalog namespace:
`cyntexa_dev.customer.customer_clean`

2. Sensitive / PII Columns

For the customer_clean table:
Column - email,phone  
Protection - Mask for analysts  

3. Unity Catalog Groups
Data Engineer Group - cyntexa_data_engineers
* Can access the customer table.
* Can see original email,phone values.

Data Analyst Group - cyntexa_data_analysts
* Can query the customer table.
* Cannot see raw PII.
* Email and phone are masked.

4. PII Protection

Column masking is applied to `email` and `phone`.

For a larger production environment, Cyntexa can use Unity Catalog ABAC with governed PII tags and centralized masking policies.

5. Access Auditing
Access is audited at two levels.
Permission audit
```sql
SHOW GRANTS ON TABLE cyntexa_dev.customer.customer_clean;
```
<!-- This identifies which users/groups currently have privileges on the table. -->

<!-- Unity Catalog activity can be investigated using:

```sql
SELECT
    event_time,
    user_identity.email AS user_email,
    service_name,
    action_name,
    request_params
FROM system.access.audit
WHERE service_name = 'unityCatalog'
ORDER BY event_time DESC;
```

This provides an audit trail of user activity, allowing Cyntexa to investigate who accessed or modified governed resources and when.

### Governance principle

The model follows:
Least Privilege + Role-Based Access + Column Masking + Auditability -->

In [0]:
%sql
-- 8 
MERGE INTO cyntexa_dev.sales.customer_address_history AS target
USING customer_address_source AS source
ON target.customer_id = source.customer_id
   AND target.is_current = true

WHEN MATCHED
AND (
    target.email != source.email
    OR target.city != source.city
    OR target.state != source.state
    OR target.status != source.status
)
THEN UPDATE SET
    target.end_date = source.effective_date,
    target.is_current = false;

In [0]:
%sql
INSERT INTO cyntexa_dev.sales.customer_address_history
(
    customer_id,
    email,
    city,
    state,
    status,
    effective_date,
    end_date,
    is_current,
    version
)
SELECT
    source.customer_id,
    source.email,
    source.city,
    source.state,
    source.status,
    source.effective_date,
    NULL,
    true,
    COALESCE(MAX(target.version), 0) + 1
FROM customer_address_source AS source
LEFT JOIN cyntexa_dev.sales.customer_address_history AS target
ON source.customer_id = target.customer_id

WHERE NOT EXISTS (
    SELECT 1
    FROM cyntexa_dev.sales.customer_address_history AS current_record
    WHERE current_record.customer_id = source.customer_id
      AND current_record.is_current = true
      AND current_record.email = source.email
      AND current_record.status = source.status
      AND current_record.city = source.city
      AND current_record.state = source.state
)
GROUP BY
    source.customer_id,
    source.email,
    source.city,
    source.state,
    source.status,
    source.effective_date;

In [0]:
# 9
from pyspark.sql.functions import col, when, lit, count, date_format

start_date = '2026-03-01'
end_date = '2026-09-01'

report_df = (
    spark.sql(f"""
        SELECT
            date_format(effective_date, 'yyyy-MM') AS month,
            COUNT(DISTINCT customer_id) AS retained_customers
        FROM cyntexa_dev.sales.customer_address_history
        WHERE effective_date >= '{start_date}'
          AND (end_date IS NULL OR end_date > effective_date)
          AND is_current = true
          AND status = 'Active'
        GROUP BY month
        ORDER BY month
    """)
)

display(report_df)